# Renderizar Motion Graphics (HyperFrames) en Colab

Este notebook te guia paso a paso para renderizar un video de Motion Graphics creado con **HyperFrames** (HTML + GSAP) a un archivo MP4, todo en la nube (Google Colab), sin instalar nada en tu PC.

**Que es HyperFrames:** un framework que convierte HTML animado (con GSAP) en video. El HTML define la composicion (elementos, duraciones) y GSAP define las animaciones. El renderizador busca un timeline pausado en `window.__timelines` y captura frame por frame con Chrome, luego codifica con FFmpeg.

**Flujo general:**
1. Preparar entorno (Node.js, Python libs, libs del sistema).
2. Subir tu proyecto (zip con index.html).
3. Renderizar con `hyperframes render`.
4. Descargar el MP4.


## Paso 1 - Instalar Node.js 22 con nvm

HyperFrames CLI esta escrito en Node.js y requiere Node 22+. Colab no trae Node por defecto, asi que usamos **nvm** (Node Version Manager) para instalarlo sin apt.


In [ ]:
# Descarga e instala nvm (gestor de versiones de Node)
!curl -o- https://raw.githubusercontent.com/nvm-sh/nvm/v0.39.7/install.sh | bash
# Instala Node 22 y lo activa; muestra versiones para verificar
!export NVM_DIR="$HOME/.nvm" && . "$NVM_DIR/nvm.sh" && nvm install 22 && nvm use 22 && node --version && npm --version


## Paso 2 - Instalar FFmpeg y Chromium con pip

- **imageio-ffmpeg**: paquete Python que descarga un binario estatico de FFmpeg (evita apt y problemas de PATH).
- **playwright**: paquete Python que descarga un Chromium compatible. Lo usaremos como navegador para el render.


In [ ]:
# Instala FFmpeg (binario estatico) y Playwright (Chromium) via pip
!pip install -q imageio-ffmpeg playwright
# Descarga el binario de Chromium que usara Playwright
!playwright install chromium
import imageio_ffmpeg, playwright
# Muestra la ruta del binario de FFmpeg para verificar
print('ffmpeg bin:', imageio_ffmpeg.get_ffmpeg_exe())


## Paso 2b - Librerias del sistema para Chrome (Ubuntu 24.04)

El Chromium descargado por HyperFrames/Playwright necesita librerias compartidas del sistema (libatk, libnss, etc.). En Ubuntu 24.04 algunas cambiaron de nombre (ej. `libasound2` -> `libasound2t64`). Esta celda las instala con apt.


In [ ]:
# Instala las librerias compartidas que necesita Chrome/Chromium en Ubuntu 24.04
!apt-get update -qq
!apt-get install -y -qq libnss3 libnspr4 libatk1.0-0 libatk-bridge2.0-0 libcups2 libdrm2 libxkbcommon0 libatspi2.0-0 libxcomposite1 libxdamage1 libxfixes3 libxrandr2 libgbm1 libpango-1.0-0 libcairo2 libasound2t64


## Paso 3 - Configurar variables de entorno

HyperFrames busca el navegador y FFmpeg mediante variables de entorno:
- `HYPERFRAMES_FFMPEG_PATH`: ruta al binario de FFmpeg.
- `HYPERFRAMES_BROWSER_PATH` / `CHROME_PATH`: ruta al Chromium que usara para capturar frames.

Preferimos el Chromium de Playwright (o el del sistema) sobre el que HyperFrames descarga, porque ya tiene las libs instaladas.


In [ ]:
# Configura rutas de FFmpeg y Chromium para HyperFrames
import os, glob
# Ruta del binario de FFmpeg (de imageio-ffmpeg)
ffmpeg_bin = imageio_ffmpeg.get_ffmpeg_exe()
os.environ['HYPERFRAMES_FFMPEG_PATH'] = ffmpeg_bin
# Busca Chromium de Playwright en su cache
chromium_candidates = glob.glob('/root/.cache/ms-playwright/chromium-*/chrome-linux/chrome') + glob.glob('/home/*/.cache/ms-playwright/chromium-*/chrome-linux/chrome')
# Si no hay Playwright, usa el Chromium del sistema
system_chromium = None
for c in ['/usr/bin/chromium','/usr/bin/chromium-browser','/usr/bin/google-chrome']:
    if os.path.exists(c):
        system_chromium = c
        break
chromium_bin = chromium_candidates[0] if chromium_candidates else system_chromium
if chromium_bin:
    os.environ['CHROME_PATH'] = chromium_bin
    os.environ['HYPERFRAMES_BROWSER_PATH'] = chromium_bin
    os.environ['PUPPETEER_EXECUTABLE_PATH'] = chromium_bin
print('FFMPEG:', os.environ.get('HYPERFRAMES_FFMPEG_PATH'))
print('CHROME:', os.environ.get('CHROME_PATH'))
print('HYPERFRAMES_BROWSER_PATH:', os.environ.get('HYPERFRAMES_BROWSER_PATH'))


## Paso 4 - Subir tu proyecto (zip)

Sube un archivo `.zip` de tu carpeta de proyecto. Debe contener al menos `index.html` (la composicion raiz de HyperFrames).


In [ ]:
# Sube el zip del proyecto desde tu PC
from google.colab import files
uploaded = files.upload()
print('Subido:', list(uploaded.keys()))


## Paso 5 - Descomprimir el proyecto

Extrae el zip en `/content/project`. Si el zip tiene una carpeta raiz, mueve su contenido hacia arriba para que `index.html` quede directamente en `/content/project`.


In [ ]:
# Descomprime el zip en /content/project, aplanando carpeta raiz si existe
import zipfile, os
zip_name = list(uploaded.keys())[0]
os.makedirs('/content/project', exist_ok=True)
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/project')
entries = os.listdir('/content/project')
if len(entries) == 1 and os.path.isdir(os.path.join('/content/project', entries[0])):
    inner = os.path.join('/content/project', entries[0])
    for f in os.listdir(inner):
        os.rename(os.path.join(inner, f), os.path.join('/content/project', f))
    os.rmdir(inner)
print('Contenido:', os.listdir('/content/project'))


## Paso 5b - Instalar HyperFrames globalmente

Instala el CLI de HyperFrames de forma global con npm. Usamos `npm install -g` (en lugar de `npx`) para evitar el prompt interactivo de confirmacion que bloquea en Colab.


In [ ]:
# Instala el CLI de HyperFrames globalmente (evita prompt interactivo de npx)
import os
nvm_dir = os.path.expanduser("~/.nvm")
install_cmd = 'export NVM_DIR="' + nvm_dir + '" && . "$NVM_DIR/nvm.sh" && nvm use 22 && npm install -g hyperframes'
!{install_cmd}


## Paso 6 - Renderizar a MP4

Ejecuta `hyperframes render`. El renderizador:
1. Carga `index.html` en Chrome headless.
2. Busca `window.__timelines['main']` (el timeline pausado de GSAP).
3. Captura cada frame (seek frame-by-frame) para garantizar determinismo.
4. Codifica los frames con FFmpeg a MP4.

La variable `HYPERFRAMES_BROWSER_PATH` se pasa inline para asegurar que HyperFrames use el Chromium correcto.


In [ ]:
# Renderiza la composicion a MP4
import os
nvm_dir = os.path.expanduser("~/.nvm")
chrome = "/usr/bin/chromium-browser"  # ajusta si tu Chromium esta en otra ruta
cmd = ('HYPERFRAMES_BROWSER_PATH=' + chrome +
       ' CHROME_PATH=' + chrome +
       ' export NVM_DIR="' + nvm_dir + '" && . "$NVM_DIR/nvm.sh" && nvm use 22 && cd /content/project && hyperframes render --output /content/output.mp4')
!{cmd}


## Paso 7 - Descargar el video

Descarga el MP4 generado a tu PC.


In [ ]:
# Descarga el MP4 resultante
from google.colab import files
files.download("/content/output.mp4")
